In [1]:
from pyspark.sql import SparkSession, functions as func

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

In [3]:
spark = SparkSession.builder.appName("MinTemperatures").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/02 19:02:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/05/02 19:02:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/05/02 19:02:05 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/05/02 19:02:05 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/05/02 19:02:05 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/05/02 19:02:05 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/05/02 19:02:05 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
25/05/02 19:02:05 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.


In [4]:
schema = StructType([
StructField("stationID", StringType(), True),
StructField("date", IntegerType(), True),
StructField("measure_type", StringType(), True),
StructField("temperature", FloatType(), True)
])

In [5]:
df = spark.read.schema(schema).csv("/opt/spark/data/1800.csv")
df.printSchema()

root
 |-- stationID: string (nullable = true)
 |-- date: integer (nullable = true)
 |-- measure_type: string (nullable = true)
 |-- temperature: float (nullable = true)



In [6]:
# Filter out all but TMIN entries
minTemps = df.filter(df.measure_type == "TMIN")

# select only stationID and temperature
stationTemps = minTemps.select("stationID", "temperature")


In [7]:
# Aggregate to find minimum temperature for every station
minTempStation = stationTemps.groupBy("stationID").min("temperature")
minTempStation.show()

+-----------+----------------+
|  stationID|min(temperature)|
+-----------+----------------+
|ITE00100554|          -148.0|
|EZE00100082|          -135.0|
+-----------+----------------+



In [8]:
# Conver temperature to fahrenheit and sort the dataset
minTempsF = minTempStation.withColumn("temperature", 
                                      func.round(func.col("min(temperature)") * 0.1 * (9.0 / 5.0) + 32.0, 2)).select("stationID", "temperature").sort("temperature")

In [10]:
results = minTempsF.collect()

In [11]:
for result in results:
    print(result)

Row(stationID='ITE00100554', temperature=5.36)
Row(stationID='EZE00100082', temperature=7.7)


In [12]:
spark.stop()